In [ ]:
%matplotlib widget

import rospy
import actionlib
from assignment_2_2024.msg import PlanningAction, PlanningGoal
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from ipywidgets import widgets, Layout
from IPython.display import display
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan

In [ ]:
class ActionClientManager:
    def __init__(self):
        self.client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
        self.client.wait_for_server()
        self.reached_targets = 0
        self.not_reached_targets = 0
        self._lock = None  # Will link to shared lock if provided

    def set_lock(self, lock):
        self._lock = lock

    def send_goal(self, x, y, done_cb):
        goal = PlanningGoal()
        goal.target_pose.pose.position.x = x
        goal.target_pose.pose.position.y = y
        self.client.send_goal(goal, done_cb=lambda status, result: self._goal_done_cb(status, result, done_cb))

    def _goal_done_cb(self, status, result, user_cb):
        if self._lock:
            with self._lock:
                if status == actionlib.GoalStatus.SUCCEEDED:
                    self.reached_targets += 1
                else:
                    self.not_reached_targets += 1
        else:
            if status == actionlib.GoalStatus.SUCCEEDED:
                self.reached_targets += 1
            else:
                self.not_reached_targets += 1
        if user_cb:
            user_cb(status, result)

    def cancel_goal(self):
        self.client.cancel_goal()
        if self._lock:
            with self._lock:
                self.not_reached_targets += 1
        else:
            self.not_reached_targets += 1

In [ ]:
class VisualizationManager:
    def __init__(self):
        self.robot_x = 0.0
        self.robot_y = 0.0
        self.min_obstacle_dist = float('inf')
        self.position_history = []
        self.reached_targets = 0
        self.not_reached_targets = 0
        self.lock = None  # For thread safety if provided

        self.fig, (self.ax1, self.ax2) = plt.subplots(1, 2, figsize=(10, 4))
        self.robot_plot, = self.ax1.plot([], [], 'b-', linewidth=2)
        self.ax1.set_title('Robot Trajectory')
        self.ax1.set_xlabel('X position')
        self.ax1.set_ylabel('Y position')
        self.ax1.grid(False)
        self.ax1.set_xlim(-10, 10)
        self.ax1.set_ylim(-10, 10)
        self.target_bars = self.ax2.bar(
            ['Reached', 'Not Reached'],
            [self.reached_targets, self.not_reached_targets],
            color=['green', 'red']
        )
        self.ax2.set_title('Target Achievement')
        self.ax2.set_ylabel('Count')
        self.ax2.set_ylim(0, 10)
        self.ani = FuncAnimation(
            self.fig, self.update_plots, interval=100, blit=True, cache_frame_data=False
        )

        self.position_output = None
        self.distance_output = None

    def set_lock(self, lock):
        self.lock = lock

    def set_widgets(self, position_output, distance_output):
        self.position_output = position_output
        self.distance_output = distance_output

    def update_robot_position(self, x, y):
        if self.lock:
            with self.lock:
                self.robot_x = x
                self.robot_y = y
                self.position_history.append((x, y))
        else:
            self.robot_x = x
            self.robot_y = y
            self.position_history.append((x, y))

    def update_min_obstacle_dist(self, dist):
        if self.lock:
            with self.lock:
                self.min_obstacle_dist = dist
        else:
            self.min_obstacle_dist = dist

    def update_target_counts(self, reached, not_reached):
        if self.lock:
            with self.lock:
                self.reached_targets = reached
                self.not_reached_targets = not_reached
        else:
            self.reached_targets = reached
            self.not_reached_targets = not_reached

    def update_plots(self, frame=None):
        if self.lock:
            with self.lock:
                x_vals = [p[0] for p in self.position_history]
                y_vals = [p[1] for p in self.position_history]
                self.robot_plot.set_data(x_vals, y_vals)
                self.ax1.relim()
                self.ax1.autoscale_view()
                for bar, height in zip(self.target_bars, [self.reached_targets, self.not_reached_targets]):
                    bar.set_height(height)
                if self.position_output:
                    self.position_output.value = f"X: {self.robot_x:.2f}, Y: {self.robot_y:.2f}"
                if self.distance_output:
                    dist = self.min_obstacle_dist if self.min_obstacle_dist != float('inf') else '∞'
                    self.distance_output.value = f"Min Obstacle Distance: {dist:.2f} m"
        else:
            x_vals = [p[0] for p in self.position_history]
            y_vals = [p[1] for p in self.position_history]
            self.robot_plot.set_data(x_vals, y_vals)
            self.ax1.relim()
            self.ax1.autoscale_view()
            for bar, height in zip(self.target_bars, [self.reached_targets, self.not_reached_targets]):
                bar.set_height(height)
            if self.position_output:
                self.position_output.value = f"X: {self.robot_x:.2f}, Y: {self.robot_y:.2f}"
            if self.distance_output:
                dist = self.min_obstacle_dist if self.min_obstacle_dist != float('inf') else '∞'
                self.distance_output.value = f"Min Obstacle Distance: {dist:.2f} m"
        return (self.robot_plot, *self.target_bars)